In [ ]:
import temet
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as colors
import glob
import os
import numpy as np
import random

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from util.sim_data_helpers import get_data_from_fof_folder, get_cosmo_parameters

plt.rcdefaults()   # undo temet's global style


def load_spectra_from_boxes(gp_paths, spectrum_num, z=2.0):
    """ Loads the spectrum with index spectrum_num (same line of sight) from every box specified in gp_paths

    Args:
        gp_paths (list): List of Gridpoint folder paths to load spectra from.
        spectrum_num (int): Index of the spectrum.
        z (float, optional): Redshift value. Defaults to 2.0.

    Returns:
        wavelengths (list): one np.array containing the wavelengths of the spectrum for each box
        fluxes (list): one np.array containing the flux of the spectrum for each box
        params (list): (Omega0, OmegaBaryon, OmegaLambda, HubbleParam) for each box
    """
    wavelengths = []
    fluxes = []
    params = []

    for gp_path in gp_paths:
        gp_name = gp_path.split("/")[-2]
        path = gp_path + f"data.files/spectra/spectra_{gp_name}_z{z:.1f}_n100d2-fullbox_SDSS-BOSS_HI_combined.hdf5"
        with h5py.File(path, "r") as f:
            wavelengths.append(f["wave"][:])
            fluxes.append(f["flux"][spectrum_num])

        params.append(get_cosmo_parameters(gp_path))

    return wavelengths, fluxes, params


def get_data_from_header(output_dir, snapN, param='BoxSize'):
    """ get data from the header of a specified simulation and snapshot
    
    Args:
        output_dir (string): base directory of the simulation
        snapN (int): number of the snapshot
        param (string): parameter of the hdf5 file (the parameter to read)

    Returns:
        np.array : Array containing the specific data
    """

    snapdir = glob.glob(output_dir+f"output/snapdir_*{snapN}")[0]
    snap_files = os.listdir(snapdir)
    file_name = snap_files[0]
    file_path = snapdir+f"/{file_name}"

    with h5py.File(file_path, "r") as f:
        header = f['Header']
        return_param = header.attrs[param]
        
    return return_param

In [ ]:
gps_list = [i for i in range(50)]

base_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite"
gp_paths = [base_path+f"/gridpoint{i}/" for i in gps_list]

spectrum_num = 2
wavelengths, fluxes, params = load_spectra_from_boxes(gp_paths, spectrum_num)

cosmo_par_index = 0 # 0: Omega0, 1: OmegaBaryon, 2: OmegaLambda, 3: HubbleParam
# order everything by the first cosmological parameter (Omega0), descending
order = np.argsort([p[cosmo_par_index] for p in params])#[::-1]
wavelengths = [wavelengths[i] for i in order]
fluxes = [fluxes[i] for i in order]
params = [params[i] for i in order]


In [ ]:
redshifts = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]
snaps = [12, 11, 10, 9, 8, 7, 6, 5, 4, 3, "002"]
gp_here = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/reference/"#gp_paths[0]  # Choose the first gridpoint for demonstration

print(gp_here)

In [ ]:
fig = plt.subplots(figsize=(4, 5))

ray_num = 2843

x_max = 1245
x_min = 1210

colors = plt.cm.plasma(np.linspace(0, 0.8, len(redshifts)))

for i in range(len(redshifts)):
    z = redshifts[i]
    offset = 10 - i
    w, y, p = load_spectra_from_boxes([gp_here], ray_num, z=z)
    z_real = get_data_from_header(gp_here, snaps[i], param="Redshift")

    x_plot = np.array(w[0])/(z_real+1)
    y_plot = y[0] + offset

    plt.text(x_min+0.5, offset+0.9, f"z={z:.1f}", verticalalignment="top", horizontalalignment="left", color=colors[i])

    plt.plot(x_plot, y_plot, c=colors[i])

plt.xlim(1210, 1245)
plt.xlabel(r"Restframe Wavelength [$\AA$]")
plt.ylabel("Relative Flux (+ constant offset)")
plt.tight_layout()

plt.savefig("plots/spectra_stack_one_los_restwavelength.pdf", format="pdf")
plt.show()

In [ ]:
fig, axe = plt.subplots(figsize=(4, 5))

cosmo_par = [i[cosmo_par_index] for i in params]

# all boxes share the same wavelength grid, so the spectra stack into one 2D array (one row per box)
flux_2d = np.vstack(fluxes)
rows = np.arange(len(fluxes))

mesh = axe.pcolormesh(wavelengths[0]/(1+2), rows, flux_2d, shading="nearest", cmap="magma")#, vmin=0, vmax=1)

axe.annotate(r"z=2.0", [1243, 0.82], fontsize=10, verticalalignment="bottom", horizontalalignment="right")

# rows are sorted by the cosmological parameter, so label a few of them with its value
ticks = np.linspace(0, len(rows)-1, 5).astype(int)
axe.set_yticks(ticks)
axe.set_yticklabels([f"{cosmo_par[t]:.2f}" for t in ticks])

cbar = plt.colorbar(mesh, ax=axe, aspect=40)   # larger aspect -> thinner bar (default 20)
cbar.set_label(r"Relative flux")
axe.set_xlabel(r"Restframe Wavelength [$\AA$]")
axe.set_ylabel(r"$\Omega_m$")
axe.set_xlim([1210, 1245])
plt.tight_layout()
plt.savefig("plots/spectra_2d_stack_boxes.pdf", format="pdf")
plt.show()